# CELL 1 —  CẤU HÌNH


In [1]:
HF_CLEAN_REPO = "datdong2004/amazonNew-cleaned"  # ← repo đích (private)

# Chọn categories muốn xử lý trong session này.
# 3 cách đặt giá trị:
#
#   "all"       → toàn bộ 28 categories chưa xử lý
#   "remaining" → tất cả categories còn lại theo checkpoint
#   [list]      → chỉ định tay, ví dụ:
#                 ["software", "gift_cards", "alexa_skills"]
#
# ── SESSION 1: 24 categories nhỏ + trung bình (~4 giờ) ─────
#TARGET_CATEGORIES = [
#    "alexa_skills", "gift_cards", "magazine_subscriptions",
#    "home_&_business_services", "collectibles_&_fine_art",
#   "digital_music", "software", "appliances",
#   "musical_instruments", "automotive", "industrial_&_scientific",
#   "patio,_lawn_&_garden", "arts,_crafts_&_sewing",
#   "cds_&_vinyl", "video_games", "office_products",
#   "grocery_&_gourmet_food", "cell_phones_&_accessories",
#   "pet_supplies", "toys_&_games", "kindle_store",
#   "sports_&_outdoors", "tools_&_home_improvement",
#    "movies_&_tv",
#]

# ── SESSION 2: 4 categories lớn nhất (~4 giờ) ───────────────
# Sau khi session 1 xong, đổi thành:
TARGET_CATEGORIES = [
    "electronics",
    "home_&_kitchen",
    "clothing,_shoes_&_jewelry",
    "books",                       # lớn nhất — để cuối cùng
]

# Thứ tự xử lý
#   "size_asc"  → nhỏ trước, lớn sau (an toàn, validate pipeline rẻ)
#   "size_desc" → lớn trước
#   "as_is"     → giữ nguyên thứ tự trong list trên
PROCESS_ORDER = "as_is"    # session 1 & 2 đều dùng as_is — thứ tự đã sắp sẵn

# Dừng toàn bộ loop nếu disk còn dưới ngưỡng này (GB)
DISK_THRESHOLD_GB = 3.0

# ── Danh sách 28 categories (tên thư mục thực tế trên HF) ──
ALL_CATEGORIES = [
    # Nhỏ — xử lý trước
    "alexa_skills", "gift_cards", "magazine_subscriptions",
    "home_&_business_services", "collectibles_&_fine_art",
    "digital_music", "software", "appliances",
    # Trung bình
    "musical_instruments", "automotive", "industrial_&_scientific",
    "patio,_lawn_&_garden", "arts,_crafts_&_sewing",
    "cds_&_vinyl", "video_games", "office_products",
    "grocery_&_gourmet_food", "cell_phones_&_accessories",
    "pet_supplies", "toys_&_games", "kindle_store",
    "sports_&_outdoors", "tools_&_home_improvement",
    # Lớn — xử lý cuối
    "movies_&_tv", "electronics", "home_&_kitchen",
    "clothing,_shoes_&_jewelry", "books",
]


# CELL 2 — Cài đặt & đăng nhập HuggingFace


In [ ]:
!pip install -q pyspark huggingface_hub

from huggingface_hub import login
login(token=os.getenv("HF_TOKEN"))


# CELL 3 — Imports & đường dẫn


In [3]:
import os, shutil, json, time
from datetime import datetime
from huggingface_hub import snapshot_download, create_repo, repo_exists
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, FloatType, TimestampType

REPO_ID      = "datdong2004/amazonNew"
RAW_DIR      = "/kaggle/working/raw"
CLEANED_DIR  = "/kaggle/working/cleaned"   # temp — push HF xong xóa
MANIFEST_DIR = "/kaggle/output/manifest"   # checkpoint (commit để lưu)
CHECKPOINT_F = f"{MANIFEST_DIR}/checkpoints.json"

for d in [RAW_DIR, CLEANED_DIR, MANIFEST_DIR, "/kaggle/working/spark_tmp"]:
    os.makedirs(d, exist_ok=True)


# CELL 4 — Checkpoint helpers


In [4]:
def load_checkpoints() -> dict:
    if os.path.exists(CHECKPOINT_F):
        with open(CHECKPOINT_F) as f:
            cp = json.load(f)
        print(f"✅ Checkpoint from /kaggle/output: {len(cp)} done")
        return cp
    try:
        snapshot_download(
            repo_id       = HF_CLEAN_REPO,
            repo_type     = "dataset",
            local_dir     = "/kaggle/working/hf_manifest",
            allow_patterns= ["manifest/checkpoints.json"],
        )
        src = "/kaggle/working/hf_manifest/manifest/checkpoints.json"
        if os.path.exists(src):
            with open(src) as f:
                cp = json.load(f)
            with open(CHECKPOINT_F, "w") as f:
                json.dump(cp, f, indent=2)
            print(f"✅ Checkpoint from HuggingFace: {len(cp)} done")
            return cp
    except Exception as e:
        print(f"ℹ️  HF checkpoint not found: {e}")
    print("ℹ️  First run — empty checkpoint.")
    return {}


def save_checkpoint(cp: dict):
    with open(CHECKPOINT_F, "w") as f:
        json.dump(cp, f, indent=2, default=str)


def push_checkpoint(cp: dict):
    """Lưu checkpoint local + push lên HF dùng upload_file."""
    from huggingface_hub import HfApi
    save_checkpoint(cp)
    api = HfApi()
    api.upload_file(
        path_or_fileobj = CHECKPOINT_F,
        path_in_repo    = "manifest/checkpoints.json",
        repo_id         = HF_CLEAN_REPO,
        repo_type       = "dataset",
        commit_message  = f"checkpoint: {len(cp)} keys",
    )

def process_category_in_batches(cat: str, batch_size: int = 10) -> tuple[int, int]:
    from huggingface_hub import HfApi
    api    = HfApi()
    folder = f"main_category={cat}"

    all_files = sorted([
        f.path for f in api.list_repo_tree(
            REPO_ID, path_in_repo=folder,
            repo_type="dataset", recursive=True
        )
        if hasattr(f, "path") and f.path.endswith(".snappy.parquet")
    ])
    total_files = len(all_files)
    batches     = [all_files[i:i+batch_size]
                   for i in range(0, total_files, batch_size)]
    print(f"  {total_files} files → {len(batches)} batches")

    # Resume sau crash
    batch_cp_key = f"{cat}__batches"
    done_batches = set(checkpoints.get(batch_cp_key, []))
    total_raw, total_clean = checkpoints.get(f"{cat}__partial", [0, 0])
    if done_batches:
        print(f"  Resuming: {len(done_batches)} batches already done")

    for b_idx, batch_files in enumerate(batches, 1):
        if b_idx in done_batches:
            print(f"  ── Batch {b_idx}/{len(batches)} SKIP")
            continue

        print(f"\n  ── Batch {b_idx}/{len(batches)} ──")
        batch_raw_dir   = f"/kaggle/working/raw/{folder}"
        batch_clean_dir = f"/kaggle/working/cleaned/{folder}"
        os.makedirs(batch_raw_dir, exist_ok=True)

        # Download
        snapshot_download(
            repo_id                = REPO_ID,
            repo_type              = "dataset",
            local_dir              = "/kaggle/working/raw",
            allow_patterns         = batch_files,
            local_dir_use_symlinks = False,
        )
        stat    = os.statvfs("/kaggle/working")
        free_gb = (stat.f_bavail * stat.f_frsize) / 1e9
        print(f"  Downloaded. Disk free: {free_gb:.1f} GB")

        # Clean
        raw_df = spark.read.parquet(batch_raw_dir)
        b_raw, b_clean = run_pipeline(raw_df, cat)
        total_raw   += b_raw
        total_clean += b_clean
        print(f"  {b_raw:,} → {b_clean:,} rows")

        # Xóa raw ngay
        shutil.rmtree(batch_raw_dir, ignore_errors=True)

        # Đổi tên TRƯỚC khi upload → tránh overwrite batch trước
        for root, dirs, files in os.walk(batch_clean_dir):
            for fname in files:
                if fname.endswith(".parquet"):
                    old = os.path.join(root, fname)
                    new = os.path.join(root,
                          fname.replace("part-", f"batch{b_idx:03d}-part-"))
                    os.rename(old, new)
        print(f"  Renamed → batch{b_idx:03d}-part-*.parquet")

        # Upload sau khi đã đổi tên
        api.upload_large_folder(
            repo_id        = HF_CLEAN_REPO,
            repo_type      = "dataset",
            folder_path    = "/kaggle/working/cleaned",
            allow_patterns = [f"{folder}/**"],
        )
        print(f"  Pushed to HF")

        # Xóa clean local
        shutil.rmtree(batch_clean_dir, ignore_errors=True)
        stat    = os.statvfs("/kaggle/working")
        free_gb = (stat.f_bavail * stat.f_frsize) / 1e9
        print(f"  Clean deleted. Disk free: {free_gb:.1f} GB")

        # Checkpoint SAU khi upload thành công
        done_batches.add(b_idx)
        checkpoints[batch_cp_key]      = list(done_batches)
        checkpoints[f"{cat}__partial"] = [total_raw, total_clean]
        push_checkpoint(checkpoints)
        print(f"  ✅ Batch {b_idx}/{len(batches)} checkpoint saved")

    # Dọn batch checkpoint sau khi xong
    checkpoints.pop(batch_cp_key,      None)
    checkpoints.pop(f"{cat}__partial", None)

    return total_raw, total_clean


checkpoints = load_checkpoints()

Fetching ... files: 0it [00:00, ?it/s]

✅ Checkpoint from HuggingFace: 28 done


In [5]:
import os
print("CHECKPOINT_F exists:", os.path.exists(CHECKPOINT_F))
print("CHECKPOINT_F path  :", CHECKPOINT_F)
print("Categories in checkpoint:", list(checkpoints.keys()))

CHECKPOINT_F exists: True
CHECKPOINT_F path  : /kaggle/output/manifest/checkpoints.json
Categories in checkpoint: ['alexa_skills', 'gift_cards', 'magazine_subscriptions', 'home_&_business_services', 'collectibles_&_fine_art', 'digital_music', 'software', 'appliances', 'musical_instruments', 'automotive', 'industrial_&_scientific', 'patio,_lawn_&_garden', 'arts,_crafts_&_sewing', 'cds_&_vinyl', 'video_games', 'office_products', 'grocery_&_gourmet_food', 'cell_phones_&_accessories', 'pet_supplies', 'toys_&_games', 'kindle_store', 'sports_&_outdoors', 'tools_&_home_improvement', 'electronics', 'home_&_kitchen', 'clothing,_shoes_&_jewelry', 'books__batches', 'books__partial']


In [6]:
# Xem checkpoint local vs HF
import json, os

# Local
if os.path.exists(CHECKPOINT_F):
    with open(CHECKPOINT_F) as f:
        local_cp = json.load(f)
    print(f"Local: {len(local_cp)} keys")
    print(sorted(local_cp.keys()))
else:
    print("Local checkpoint không tồn tại")

Local: 28 keys
['alexa_skills', 'appliances', 'arts,_crafts_&_sewing', 'automotive', 'books__batches', 'books__partial', 'cds_&_vinyl', 'cell_phones_&_accessories', 'clothing,_shoes_&_jewelry', 'collectibles_&_fine_art', 'digital_music', 'electronics', 'gift_cards', 'grocery_&_gourmet_food', 'home_&_business_services', 'home_&_kitchen', 'industrial_&_scientific', 'kindle_store', 'magazine_subscriptions', 'musical_instruments', 'office_products', 'patio,_lawn_&_garden', 'pet_supplies', 'software', 'sports_&_outdoors', 'tools_&_home_improvement', 'toys_&_games', 'video_games']



# CELL 5 — Resolve danh sách categories sẽ chạy session này


In [7]:
done_set = set(
    k for k in checkpoints.keys()
    if not k.endswith("__batches") and not k.endswith("__partial")
)

if TARGET_CATEGORIES == "all":
    queue = [c for c in ALL_CATEGORIES if c not in done_set]
elif TARGET_CATEGORIES == "remaining":
    queue = [c for c in ALL_CATEGORIES if c not in done_set]
else:
    # List chỉ định tay — bỏ qua cái đã xong
    invalid = [c for c in TARGET_CATEGORIES if c not in ALL_CATEGORIES]
    if invalid:
        raise ValueError(f"❌ Không nhận diện được: {invalid}")
    queue = [c for c in TARGET_CATEGORIES if c not in done_set]
    already = [c for c in TARGET_CATEGORIES if c in done_set]
    if already:
        print(f"⚠️  Bỏ qua (đã xong): {already}")

# Sắp xếp theo thứ tự đã chọn
if PROCESS_ORDER == "size_asc":
    queue = sorted(queue, key=lambda c: ALL_CATEGORIES.index(c))
elif PROCESS_ORDER == "size_desc":
    queue = sorted(queue, key=lambda c: ALL_CATEGORIES.index(c), reverse=True)
# "as_is" giữ nguyên

print(f"\n── Session này sẽ xử lý {len(queue)} categories ──────────")
for i, c in enumerate(queue, 1):
    print(f"  {i:>2}. {c}")
print(f"\n── Đã xong từ trước: {sorted(done_set)}")
print(f"── Còn lại sau session: "
      f"{[c for c in ALL_CATEGORIES if c not in done_set and c not in queue]}")


⚠️  Bỏ qua (đã xong): ['electronics', 'home_&_kitchen', 'clothing,_shoes_&_jewelry']

── Session này sẽ xử lý 1 categories ──────────
   1. books

── Đã xong từ trước: ['alexa_skills', 'appliances', 'arts,_crafts_&_sewing', 'automotive', 'cds_&_vinyl', 'cell_phones_&_accessories', 'clothing,_shoes_&_jewelry', 'collectibles_&_fine_art', 'digital_music', 'electronics', 'gift_cards', 'grocery_&_gourmet_food', 'home_&_business_services', 'home_&_kitchen', 'industrial_&_scientific', 'kindle_store', 'magazine_subscriptions', 'musical_instruments', 'office_products', 'patio,_lawn_&_garden', 'pet_supplies', 'software', 'sports_&_outdoors', 'tools_&_home_improvement', 'toys_&_games', 'video_games']
── Còn lại sau session: ['movies_&_tv']



# CLEANING STAGES (dùng chung cho mọi category)


In [8]:
def stage1_filter_nulls(df: DataFrame) -> DataFrame:
    return df.filter(
        F.col("reviewText").isNotNull() &
        F.col("asin").isNotNull()       &
        F.col("reviewerID").isNotNull() &
        (F.length("reviewText") > 10)
    )

def stage2_exact_dedup(df: DataFrame) -> DataFrame:
    return df.dropDuplicates(["asin", "reviewerID", "unixReviewTime"])

def stage3_near_dedup(df: DataFrame) -> DataFrame:
    return (df
        .withColumn("_fp", F.md5(F.substring("reviewText", 1, 200)))
        .dropDuplicates(["reviewerID", "_fp"])
        .drop("_fp")
    )

def stage4_type_casting(df: DataFrame) -> DataFrame:
    # Extract số từ dạng range: "$1.5 - $3.0" → low=1.5, high=3.0
    low  = F.regexp_extract("price", r"(\d+\.?\d*)\s*[-–]\s*\d+\.?\d*", 1)
    high = F.regexp_extract("price", r"\d+\.?\d*\s*[-–]\s*(\d+\.?\d*)", 1)

    price_clean = F.when(
        # Case 1: dạng range → trung bình
        (low != "") & (high != ""),
        (low.cast(FloatType()) + high.cast(FloatType())) / 2
    ).when(
        # Case 2: số đơn → lấy số đầu tiên tìm được
        F.regexp_extract("price", r"(\d+\.?\d*)", 1) != "",
        F.regexp_extract("price", r"(\d+\.?\d*)", 1).cast(FloatType())
    ).otherwise(
        # Case 3: không extract được → null
        F.lit(None).cast(FloatType())
    )

    return df.withColumns({
        "overall": F.when(
            F.col("overall").cast(IntegerType()).between(1, 5),
            F.col("overall").cast(IntegerType())
        ).otherwise(F.lit(None)),

        "review_date": F.when(
            F.col("unixReviewTime").between(788_918_400, 1_893_456_000),
            F.from_unixtime("unixReviewTime").cast(TimestampType())
        ).otherwise(F.lit(None)),

        "price_clean": price_clean,
    }).drop("unixReviewTime")
   


def stage4b_handle_malformed(df: DataFrame) -> DataFrame:
    """
    Phát hiện và xử lý dữ liệu malformed trong các cột text và ID.

    Các loại malformed được xử lý:
      - reviewText/text chứa ký tự encoding rác (replacement char \ufffd)
        → thay bằng null, sẽ bị lọc ở stage1 nếu xảy ra trước
        → flag nếu xảy ra sau stage1
      - asin sai format (không phải 10 ký tự alphanum)
        → flag, không xóa (asin vẫn có thể dùng để join)
      - reviewerID quá ngắn (< 5 ký tự) → flag
      - reviewText chứa > 80% ký tự không phải ASCII
        → thường là encoding lỗi hoặc ngôn ngữ không hỗ trợ → flag
    """
    # ── Xử lý encoding rác trong text columns ───────────────
    for col in ["reviewText", "text", "summary"]:
        if col in df.columns:
            df = df.withColumn(col,
                # Thay replacement character (dấu hiệu encoding lỗi) → null
                F.when(
                    F.col(col).contains("\ufffd"),
                    F.lit(None)
                ).otherwise(F.col(col))
            )

    # ── Flag malformed IDs ───────────────────────────────────
    df = df.withColumns({
        # asin chuẩn: 10 ký tự chữ/số (B + 9 ký tự)
        "flag_malformed_asin": ~F.col("asin").rlike(r"^[A-Z0-9]{10}$"),

        # reviewerID quá ngắn
        "flag_malformed_reviewer": F.length("reviewerID") < 5,

        # reviewText có thể là encoding lỗi:
        # tỉ lệ ký tự non-ASCII > 80%
        "flag_non_ascii_review": (
            F.length(F.regexp_replace("reviewText", r"[^\x00-\x7F]", ""))
              .cast(FloatType()) /
            F.length("reviewText").cast(FloatType())
        ) < 0.2,   # < 20% ASCII → > 80% non-ASCII

        # overall bị null sau casting (giá trị ngoài [1,5])
        "flag_malformed_overall": F.col("overall").isNull(),

        # review_date bị null sau casting (timestamp không hợp lệ)
        "flag_malformed_date": F.col("review_date").isNull(),
    })

    return df

def stage5_clean_text(df: DataFrame) -> DataFrame:
    # reviewText — aggressive (lowercase, strip noise)
    df = (df
        .withColumn("reviewText",
            F.regexp_replace("reviewText", r"<[^>]+>", " "))
        .withColumn("reviewText",
            F.regexp_replace("reviewText", r"http\S+|www\.\S+", ""))
        .withColumn("reviewText",
            F.regexp_replace("reviewText", r"[^\w\s.,!?']", " "))
        .withColumn("reviewText",
            F.regexp_replace("reviewText", r"\s{2,}", " "))
        .withColumn("reviewText",
            F.lower(F.trim("reviewText")))
    )
    # text (product desc) — moderate (giữ case, giữ structure)
    df = (df
        .withColumn("text",
            F.regexp_replace("text", r"<[^>]+>", " "))
        .withColumn("text",
            F.regexp_replace("text", r"http\S+|www\.\S+", ""))
        .withColumn("text",
            F.regexp_replace("text", r"\s{2,}", " "))
        .withColumn("text", F.trim("text"))
    )
    return df

def stage6_clean_metadata(df: DataFrame) -> DataFrame:
    for col in ["title", "summary", "brand"]:
        if col in df.columns:
            df = df.withColumn(col,
                F.trim(F.regexp_replace(col, r"<[^>]+>", "")))
    return df.drop("reviewerName")

def stage7_flag_and_write(df: DataFrame, cat: str) -> int:
    if "main_category" not in df.columns:
        df = df.withColumn("main_category", F.lit(cat))
    df = df.withColumns({
        "flag_short_review":      F.length("reviewText") < 30,
        "flag_no_prod_desc":      F.col("text").isNull() | (F.length("text") < 20),
        "flag_missing_price":     F.col("price_clean").isNull(),
        "flag_extreme_rating":    F.col("overall").isin(1, 5),
        # flag malformed đã được thêm trong stage4b, giữ nguyên không ghi đè
    })
    out = os.path.join(CLEANED_DIR, f"main_category={cat}")
    (df.repartition(8)
       .write.mode("overwrite")
       .partitionBy("overall")
       .parquet(out)
    )
    return spark.read.parquet(out).count()

def run_pipeline(df: DataFrame, cat: str) -> tuple[int, int]:
    """Chạy 7 stages, trả về (raw_count, clean_count)."""
    raw = df.count()
    df = stage1_filter_nulls(df)
    df = stage2_exact_dedup(df)
    df = stage3_near_dedup(df)
    df = stage4_type_casting(df)
    df = stage4b_handle_malformed(df)
    df = stage5_clean_text(df)
    df = stage6_clean_metadata(df)
    clean = stage7_flag_and_write(df, cat)
    return raw, clean


# CELL 6 — Spark Session 


In [9]:
spark = (SparkSession.builder
    .appName("AmazonReviewsCleaning")
    .master("local[4]")
    .config("spark.driver.memory",                           "12g")
    .config("spark.driver.maxResultSize",                    "4g")
    .config("spark.sql.shuffle.partitions",                  "64")
    .config("spark.local.dir",                               "/tmp/spark_tmp")  # /tmp không giới hạn quota như /kaggle/working
    .config("spark.sql.parquet.compression.codec",           "zstd")
    .config("spark.sql.adaptive.enabled",                    "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} ready")

# Tạo HF repo nếu chưa tồn tại
if not repo_exists(HF_CLEAN_REPO, repo_type="dataset"):
    create_repo(HF_CLEAN_REPO, repo_type="dataset", private=True)
    print(f"✅ Created HF repo: https://huggingface.co/datasets/{HF_CLEAN_REPO}")
else:
    print(f"✅ HF repo exists: https://huggingface.co/datasets/{HF_CLEAN_REPO}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/09 12:15:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/09 12:15:35 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in standalone/kubernetes and LOCAL_DIRS in YARN).


Spark 4.0.2 ready
✅ HF repo exists: https://huggingface.co/datasets/datdong2004/amazonNew-cleaned


In [10]:
from huggingface_hub import repo_exists
if not repo_exists(HF_CLEAN_REPO, repo_type="dataset"):
    create_repo(HF_CLEAN_REPO, repo_type="dataset", private=True)
    print(f"✅ Created HF repo: https://huggingface.co/datasets/{HF_CLEAN_REPO}")
else:
    print(f"✅ HF repo exists: https://huggingface.co/datasets/{HF_CLEAN_REPO}")

✅ HF repo exists: https://huggingface.co/datasets/datdong2004/amazonNew-cleaned



# CELL 7 — Main loop: xử lý nhiều categories tuần tự
Disk flow mỗi vòng lặp:
download raw → clean → push HF → xóa raw + cleaned local
→ disk sạch → download category tiếp theo


In [11]:
session_results = []
t_session = time.time()

for idx, cat in enumerate(queue, 1):
    folder    = f"main_category={cat}"
    raw_dir   = os.path.join(RAW_DIR,     folder)
    clean_dir = os.path.join(CLEANED_DIR, folder)

    print(f"\n{'='*62}")
    print(f"  [{idx}/{len(queue)}] {folder}")
    print(f"{'='*62}")

    # ── Step 1: Dọn cache + kiểm tra disk ────────────────────
    for tmp in [os.path.join(RAW_DIR, ".cache"),
                "/kaggle/working/hf_manifest",
                "/kaggle/working/upload_staging",
                "/kaggle/working/upload_staging_cp"]:
        if os.path.exists(tmp):
            shutil.rmtree(tmp)

    stat    = os.statvfs("/kaggle/working")
    free_gb = (stat.f_bavail * stat.f_frsize) / 1e9
    print(f"  Disk free: {free_gb:.1f} GB")
    if free_gb < DISK_THRESHOLD_GB:
        print(f"  ⛔ Disk < {DISK_THRESHOLD_GB}GB — dừng loop.")
        break

    t_cat = time.time()

    # ── Step 2: Đếm file trên HF (không download) ────────────
    from huggingface_hub import HfApi
    api = HfApi()
    remote_files = sorted([
        f.path for f in api.list_repo_tree(
            REPO_ID, path_in_repo=folder,
            repo_type="dataset", recursive=True
        )
        if hasattr(f, "path") and f.path.endswith(".snappy.parquet")
    ])
    n_files = len(remote_files)
    print(f"  Remote files: {n_files}")

    # ── Step 3: Batch hoặc single mode ───────────────────────
    LARGE_THRESHOLD = 20

    if n_files > LARGE_THRESHOLD:
        print(f"  Large category ({n_files} files) → batch mode")
        raw_count, clean_count = process_category_in_batches(cat, batch_size=80)
        skip_cleanup = True
    else:
        print(f"  Normal ({n_files} files) → single mode, downloading...")
        snapshot_download(
            repo_id                = REPO_ID,
            repo_type              = "dataset",
            local_dir              = RAW_DIR,
            allow_patterns         = [f"{folder}/*.snappy.parquet"],
            local_dir_use_symlinks = False,
        )
        raw_df = spark.read.parquet(raw_dir)
        raw_count, clean_count = run_pipeline(raw_df, cat)
        skip_cleanup = False

    removed     = raw_count - clean_count
    removed_pct = round(removed / raw_count * 100, 2) if raw_count > 0 else 0
    print(f"  Total: {raw_count:,} → {clean_count:,} (-{removed_pct}%)")

    # ── Step 4: Xóa raw (chỉ single mode) ────────────────────
    if not skip_cleanup:
        shutil.rmtree(raw_dir, ignore_errors=True)
        stat    = os.statvfs("/kaggle/working")
        free_gb = (stat.f_bavail * stat.f_frsize) / 1e9
        print(f"  Raw deleted. Disk free: {free_gb:.1f} GB")

    # ── Step 5: Validate (chỉ single mode) ───────────────────
    if not skip_cleanup:
        print("\n  ── Validation ──")
        val_df = spark.read.parquet(clean_dir)
        val_df.select("asin", "reviewerID", "overall",
                      "review_date", "reviewText", "price_clean") \
              .show(3, truncate=60)
        val_df.select([
            F.count(F.when(F.col(c).isNull(), c)).alias(c)
            for c in ["reviewText", "overall", "review_date", "price_clean"]
        ]).show()

    # ── Step 6: Push HF (chỉ single mode) ────────────────────
    if not skip_cleanup:
        print(f"  Pushing to HuggingFace...")
        t_push = time.time()
        api.upload_large_folder(
            repo_id        = HF_CLEAN_REPO,
            repo_type      = "dataset",
            folder_path    = CLEANED_DIR,
            allow_patterns = [f"{folder}/**"],
        )
        elapsed_push = round(time.time() - t_push, 1)
        print(f"  Pushed in {elapsed_push}s")

    # ── Step 7: Checkpoint ────────────────────────────────────
    elapsed_cat = round(time.time() - t_cat, 1)
    checkpoints[cat] = {
        "raw_rows":    raw_count,
        "clean_rows":  clean_count,
        "removed_pct": removed_pct,
        "n_files":     n_files,
        "elapsed_sec": elapsed_cat,
        "hf_path":     f"hf://datasets/{HF_CLEAN_REPO}/{folder}/",
        "timestamp":   datetime.utcnow().isoformat(),
    }
    push_checkpoint(checkpoints)
    print(f"  ✅ Checkpoint saved ({len(checkpoints)}/{len(ALL_CATEGORIES)})")

    # ── Step 8: Xóa clean_dir (chỉ single mode) ──────────────
    if not skip_cleanup:
        shutil.rmtree(clean_dir, ignore_errors=True)

    stat    = os.statvfs("/kaggle/working")
    free_gb = (stat.f_bavail * stat.f_frsize) / 1e9
    print(f"  Done. Disk free: {free_gb:.1f} GB | Time: {elapsed_cat}s")

    session_results.append({
        "category": cat,
        "raw":      raw_count,
        "clean":    clean_count,
        "pct":      removed_pct,
        "time":     elapsed_cat,
    })


  [1/1] main_category=books
  Disk free: 20.9 GB
  Remote files: 150
  Large category (150 files) → batch mode
  150 files → 2 batches
  Resuming: 1 batches already done
  ── Batch 1/2 SKIP

  ── Batch 2/2 ──


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:202: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Fetching ... files: 0it [00:00, ?it/s]

  Downloaded. Disk free: 12.3 GB


  7,831,043 → 7,448,832 rows
  Renamed → batch002-part-*.parquet


...

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

  Pushed to HF
  Clean deleted. Disk free: 20.9 GB
  ✅ Batch 2/2 checkpoint saved
  Total: 16,780,755 → 15,948,443 (-4.96%)


/tmp/ipykernel_17/1390511.py:108: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp":   datetime.utcnow().isoformat(),


  ✅ Checkpoint saved (27/28)
  Done. Disk free: 20.9 GB | Time: 1549.5s



# CELL 8 — Tổng kết session


In [12]:
total_time = round(time.time() - t_session, 1)
done_total = set(checkpoints.keys())
remaining  = [c for c in ALL_CATEGORIES if c not in done_total]

print(f"\n{'='*62}")
print(f"  SESSION SUMMARY — {len(session_results)} categories processed")
print(f"{'='*62}")
print(f"  {'Category':<40} {'Raw':>10} {'Clean':>10} {'-%':>6} {'Time':>7}")
print(f"  {'-'*40} {'-'*10} {'-'*10} {'-'*6} {'-'*7}")
for r in session_results:
    print(f"  {r['category']:<40} {r['raw']:>10,} {r['clean']:>10,} "
          f"{r['pct']:>5}% {r['time']:>6}s")
print(f"\n  Total time        : {total_time}s")
print(f"  Done (all time)   : {len(done_total)}/{len(ALL_CATEGORIES)}")
print(f"  Remaining         : {remaining}")
print(f"\n  HF dataset: https://huggingface.co/datasets/{HF_CLEAN_REPO}")
if remaining:
    print(f"\n  Session tiếp theo, đặt TARGET_CATEGORIES = 'remaining'")
else:
    print(f"\n  ✅ Toàn bộ 28 categories đã xử lý xong!")
print(f"\n⚠️  COMMIT notebook để lưu checkpoint trong /kaggle/output!")

spark.stop()


  SESSION SUMMARY — 1 categories processed
  Category                                        Raw      Clean     -%    Time
  ---------------------------------------- ---------- ---------- ------ -------
  books                                    16,780,755 15,948,443  4.96% 1549.5s

  Total time        : 1550.2s
  Done (all time)   : 27/28
  Remaining         : ['movies_&_tv']

  HF dataset: https://huggingface.co/datasets/datdong2004/amazonNew-cleaned

  Session tiếp theo, đặt TARGET_CATEGORIES = 'remaining'

⚠️  COMMIT notebook để lưu checkpoint trong /kaggle/output!


In [13]:
import shutil, os

# 1. Xóa raw (category bị lỗi chưa được xóa)
shutil.rmtree("/kaggle/working/raw", ignore_errors=True)
os.makedirs("/kaggle/working/raw", exist_ok=True)

# 2. Xóa Spark shuffle temp
shutil.rmtree("/kaggle/working/spark_tmp", ignore_errors=True)
os.makedirs("/kaggle/working/spark_tmp", exist_ok=True)

# 3. Xóa HF cache nếu còn
shutil.rmtree("/kaggle/working/hf_manifest", ignore_errors=True)

# Kiểm tra lại
import subprocess
r = subprocess.run("du -sh /kaggle/working/*", 
                   shell=True, capture_output=True, text=True)
print(r.stdout)

r2 = subprocess.run(["df", "-h", "/kaggle/working"],
                    capture_output=True, text=True)
print(r2.stdout)

92K	/kaggle/working/__notebook__.ipynb
372K	/kaggle/working/cleaned
4.0K	/kaggle/working/raw
4.0K	/kaggle/working/spark_tmp

Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G  600K   20G   1% /kaggle/working

